In [7]:
%pip install pyspark==3.4.1

     ---------------------------------------- 0.0/310.8 MB ? eta -:--:--
     ---------------------------------------- 0.8/310.8 MB 6.7 MB/s eta 0:00:47
     ---------------------------------------- 1.8/310.8 MB 5.9 MB/s eta 0:00:53
     ---------------------------------------- 3.1/310.8 MB 5.9 MB/s eta 0:00:52
      --------------------------------------- 4.2/310.8 MB 6.1 MB/s eta 0:00:50
      --------------------------------------- 5.2/310.8 MB 5.6 MB/s eta 0:00:55
      --------------------------------------- 6.6/310.8 MB 5.7 MB/s eta 0:00:54
      --------------------------------------- 7.3/310.8 MB 5.7 MB/s eta 0:00:53
      --------------------------------------- 7.3/310.8 MB 5.7 MB/s eta 0:00:53
      --------------------------------------- 7.3/310.8 MB 5.7 MB/s eta 0:00:53
      --------------------------------------- 7.3/310.8 MB 5.7 MB/s eta 0:00:53
     - -------------------------------------- 8.4/310.8 MB 3.8 MB/s eta 0:01:20
     - -------------------------------------- 8

In [1]:
from pyspark.sql import SparkSession
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator

In [3]:
spark = SparkSession.builder.appName("MovieCF").getOrCreate()

In [6]:
ratings = spark.read.parquet("../datasets/full_train_dataset_with_embeddings.parquet").select(
    "UserID", "MovieID", "Rating"
)

In [7]:
train, test = ratings.randomSplit([0.8, 0.2], seed=42)

In [8]:
als = ALS(
    maxIter=10,
    regParam=0.1,
    rank=20,
    userCol="UserID",
    itemCol="MovieID",
    ratingCol="Rating",
    coldStartStrategy="drop",
)

model = als.fit(train)

In [10]:
preds = model.transform(test)
evaluator = RegressionEvaluator(
    metricName="rmse", labelCol="Rating", predictionCol="prediction"
)
rmse = evaluator.evaluate(preds)
print(f"Test RMSE = {rmse:.4f}")

Test RMSE = 0.8663
